# Car Price Studio — full-data MiniLM GPU training

**Model:** `sentence-transformers/all-MiniLM-L6-v2`, pinned revision `1110a243fdf4706b3f48f1d95db1a4f5529b4d41`.

**Before Run All:** choose **Settings → Accelerator → GPU** and enable **Internet**. A single available GPU is sufficient; this code uses `cuda:0`, not distributed training. Account verification or GPU availability may affect access.

The notebook embeds the project source, downloads the public dataset, prepares **100%** of the raw rows and runs three-fold description-price regression with two epochs per fit. It keeps make/model categorical, audits training labels with Cleanlab using structured vehicle facts and observed log prices, then selects Huber LightGBM by validation dollar MAE. Test evaluation precedes a full eligible-data refit. Descriptions are windowed at 256 tokens, up to 8,000 cleaned characters.

**Data meaning:** historical US asking prices from April–May 2021, not completed sale prices. No new MiniLM quality score is claimed until training finishes.

**Session recovery:** each optimizer fit is checkpointed. This notebook ends the training subprocess after eight hours, preserves progress in outputs, and completes successfully with `pending` status if more sessions are needed. To continue, save the version/output, attach that prior notebook output with **Add Input**, and rerun this notebook with the same settings and accelerator. This is recovery between saved runs, not automatic unlimited GPU scheduling.

Cleanlab requires a fresh output directory. Do not resume an Isolation Forest run under changed code. Existing installed weights keep their original filter and scores.


In [ ]:
import os, sys, subprocess, time, json, hashlib, base64, io, zipfile, shutil, signal
from pathlib import Path
SESSION_STARTED = time.monotonic()
WORK = Path('/kaggle/working')
assert WORK.exists(), 'Run this notebook inside Kaggle.'
ROOT = WORK/'pricing'
ROOT.mkdir(exist_ok=True)
# Preserve Kaggle's GPU-ready PyTorch build; install only project dependencies.
subprocess.run([sys.executable,'-m','pip','install','--quiet',
    'numpy>=1.26,<3','pandas>=2.1,<3','scikit-learn>=1.4,<2',
    'transformers>=5.0,<6','huggingface-hub>=1.0,<2','safetensors>=0.4',
    'cleanlab==2.9.0','lightgbm>=4.5,<5','streamlit>=1.40,<2'],check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings before training.'
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,6), 'A recent GPU PyTorch environment (>=2.6) is required.'
print('GPU:',torch.cuda.get_device_name(0),'| PyTorch:',torch.__version__)


In [ ]:
# Frozen source bundle: independent of GitHub sign-in and future repository changes.
SOURCE_SHA256 = '0f2ab862a0ca8842fd49e776c87dfcc92332e672a5988c2f7450cb9c62aba387'
SOURCE_ZIP = '...'
raw=base64.b64decode(SOURCE_ZIP)
assert hashlib.sha256(raw).hexdigest()==SOURCE_SHA256
with zipfile.ZipFile(io.BytesIO(raw)) as z:z.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'artifacts/car-price-studio-kaggle-cleanlab'
DATA=ROOT/'training-cache/full-cohort.jsonl'
print('Project ready:',ROOT)


## Restore a previous GPU session, when attached

The newest saved `pricing` output directory attached under `/kaggle/input` is copied before training. In an existing interactive session, the current local progress takes priority. The run refuses mismatched data/configuration checkpoints instead of silently mixing runs. Start a separate notebook with no old outputs attached when intentionally changing training settings.

In [ ]:
if not (OUTPUT/'training-config.json').exists():
    previous=list(Path('/kaggle/input').rglob('car-price-studio-kaggle/training-config.json'))
    if len(previous)>1:
        raise RuntimeError('Attach exactly one prior Car Price Studio output to avoid ambiguous recovery.')
    if previous:
        old_output=previous[0].parent
        shutil.copytree(old_output,OUTPUT,dirs_exist_ok=True)
        old_root=old_output.parent.parent
        old_data=old_root/'training-cache/full-cohort.jsonl'
        if old_data.exists():
            DATA.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(old_data,DATA)
            meta=old_data.with_suffix('.meta.json')
            if meta.exists():shutil.copy2(meta,DATA.with_suffix('.meta.json'))
        print('Restored:',old_output)
    else:
        print('Fresh GPU run; no prior output attached.')
else:
    print('Using existing progress in this session.')


## Prepare the entire dataset

If the Craigslist dataset is attached as a Kaggle input, its `vehicles.csv` is used. Otherwise, the notebook downloads the public Zenodo snapshot. Preparation scans all raw rows with bounded memory, removes invalid targets and duplicates, then assigns chronological train/validation/test partitions. No percentage sample is taken.

In [ ]:
if not DATA.exists():
    archive=ROOT/'training-cache/vehicles.zip'
    archive.parent.mkdir(parents=True,exist_ok=True)
    attached=list(Path('/kaggle/input').rglob('vehicles.csv'))
    if attached:
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_STORED) as z:z.write(attached[0],'vehicles.csv')
        print('Using attached dataset:',attached[0])
    elif not archive.exists():
        import urllib.request
        url='https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1'
        partial=archive.with_suffix('.part')
        urllib.request.urlretrieve(url,partial)
        partial.replace(archive)
    with zipfile.ZipFile(archive) as z:
        assert 'vehicles.csv' in z.namelist(), 'The downloaded archive is not the expected dataset.'
    subprocess.run([sys.executable,'-u','-m','scripts.prepare_full',str(archive),
        '--output',str(DATA),'--workers','2'],check=True)
    archive.unlink(missing_ok=True)
meta=json.loads(DATA.with_suffix('.meta.json').read_text())
assert meta['fraction']==1.0 and meta['raw_rows']==426880, 'Expected the entire original dataset.'
assert meta['canonical_rows']==211844, 'Eligible row count differs from the verified dataset.'
print(json.dumps(meta,indent=2))


## Train and evaluate

Run settings stay fixed across resumed sessions: three folds, two epochs, batch size four, accumulation two. Transformer weights are trainable; this is direct description-to-log-price regression, not frozen embeddings. LightGBM receives the resulting scalar dollar-price feature plus structured fields.

During assessment, validation/test targets never enter description fits. Cleanlab audits training prices using structured fields, internal CV and uncertainty estimation. Text-price features do not enter auditing. The held-out test stays intact. After assessment, a fresh OOF pass and final refit cover all eligible data, subject to that outlier filter. Checkpoints in the output permit exact optimizer recovery on the same software/hardware setup.

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'-u','-m','pricing.train',
    '--data',str(DATA),'--output',str(OUTPUT),'--device','cuda',
    '--threads','4','--folds','3','--epochs','2','--batch-size','4','--accumulation','2']
# Eight-hour total notebook budget leaves time to save outputs before platform limits.
remaining=max(0,8*3600-(time.monotonic()-SESSION_STARTED))
log_path=OUTPUT/'training.log'
run_returncode=None
if (OUTPUT/'manifest.json').exists():
    print('A completed model is already present; training skipped.')
elif remaining<300:
    print('Preparation consumed this session budget. Save outputs and resume in a new GPU session.')
else:
    environment=os.environ.copy()
    environment.update(OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',OPENBLAS_NUM_THREADS='1',TOKENIZERS_PARALLELISM='false')
    with log_path.open('a',buffering=1) as log:
        process=subprocess.Popen(cmd,cwd=ROOT,env=environment,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        deadline=time.monotonic()+remaining
        try:
            while process.poll() is None and time.monotonic()<deadline:
                time.sleep(30)
                print('\n'.join(log_path.read_text(errors='replace').splitlines()[-3:]),flush=True)
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGINT)
                try:process.wait(timeout=120)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGTERM)
                    process.wait(timeout=30)
                print('Session budget reached; latest atomic checkpoints retained.')
            else:
                run_returncode=process.returncode
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGINT)
            raise
    if run_returncode not in (None,0):
        print('Training error; preserved output for diagnosis. Last log lines:')
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-25:]))
status={'status':'ready' if (OUTPUT/'manifest.json').exists() else 'pending',
    'returncode':run_returncode,'model':'sentence-transformers/all-MiniLM-L6-v2',
    'dataset_fraction':1.0,'eligible_rows':meta['canonical_rows'],
    'resume':'Attach this saved notebook output as input and Run All with identical settings.'}
(WORK/'training-status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))


## Results and deployment bundle

`pending` means the run needs another saved GPU session or diagnosis; it is not a completed model. Assessment metrics may become available before the final deployment refit completes. A ready bundle contains the dynamic MiniLM description-price model, LightGBM, schema and manifest. Download the ZIP from Kaggle's Output pane and extract it into your Pricing repository to enable the enhanced GUI mode.

In [ ]:
report=OUTPUT/'evaluation.json'
if report.exists():
    result=json.loads(report.read_text())
    print('HELD-OUT ASSESSMENT (before final refit)')
    print(json.dumps({k:result[k] for k in ['rows','selected','stacked','description_only','note']},indent=2))
else:
    print('No completed assessment yet; no MiniLM accuracy claim.')
if (OUTPUT/'manifest.json').exists():
    from pricing.inference import PricePipeline
    pipeline=PricePipeline(OUTPUT,device='cpu')
    estimate=pipeline.predict('Well maintained used sedan with service records, clean interior and no known mechanical faults.',
        {'year':2015,'odometer':95000,'manufacturer':'toyota','model':'corolla','condition':'good',
         'fuel':'gas','transmission':'automatic','title_status':'clean'})
    print('Inference smoke estimate:',estimate)
    destination=WORK/'Car_Price_Studio_Trained.zip'
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ['app.py','requirements-training.txt','README.md']:
            z.write(ROOT/name,name)
        for file in (ROOT/'pricing').glob('*.py'):z.write(file,str(file.relative_to(ROOT)))
        z.write(ROOT/'scripts/common.py','scripts/common.py')
        for file in OUTPUT.rglob('*'):
            if not file.is_file() or 'optimizer-checkpoints' in file.parts or 'description-oof' in file.parts:continue
            if 'assessment' in file.relative_to(OUTPUT).parts:continue
            z.write(file,str(Path('artifacts/car-price-studio-cleanlab')/file.relative_to(OUTPUT)))
    print('READY — download:',destination)
else:
    print('PENDING — save this version with outputs, attach its output in a new session, and resume.')
